# Background

Keystroke dynamics is a behavioral biometric approach that analyzes typing patterns to understand user behavior. Previous studies have shown that psychological conditions, such as stress, can influence typing speed, typing consistency, and error correction frequency.

This project integrates keystroke data from the KeyRecs and Aalto datasets to create a unified session-based dataset for stress level prediction. Raw keystroke events are transformed into meaningful behavioral features that can be used for machine learning applications.

# Problem Statement

Traditional stress assessment methods often rely on questionnaires, physiological sensors, or manual observation, which may not be available in real-time scenarios.

The main challenges addressed in this project are:

- How can stress levels be estimated using only typing behavior?
- Which keystroke features are most indicative of stress?
- How can a sufficiently large and diverse dataset be constructed for predictive modeling?

# Objectives

The objectives of this preprocessing stage are:

1. Combine the KeyRecs and Aalto datasets into a unified schema.
2. Extract session-level keystroke features.
3. Generate a machine-learning-ready dataset for stress prediction.
4. Ensure the dataset satisfies the minimum requirements for sessions and unique users.
5. Improve representation of minority stress ranges through controlled synthetic augmentation.

#Import Dataset

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

#Feature Engineering

##Keyrecs

###Inter Key Timings Keyrecs

In [ ]:
def create_inter_key_timings_keyrecs(group):
    inter_keys = group["DD"].abs().tolist()
    inter_key_string = ",".join(map(lambda x: str(int(x * 1000)), inter_keys[:50]))

    return inter_keys, inter_key_string

###WPM Keyrecs

In [ ]:
def calculate_wpm_keyrecs(group):
    total_chars = len(group)
    total_time_sec = (group["DD"].abs().sum())
    total_time_min = (total_time_sec / 60)

    if total_time_min == 0:
        return 0

    wpm = ((total_chars / 5) / total_time_min)

    return round(wpm, 1)

###Backspace Rate Keyrecs

In [ ]:
def calculate_backspace_rate_keyrecs(group):
    backspace_count = ((group["key1"].astype(str).str.lower() == "backspace") | (group["key2"].astype(str).str.lower() == "backspace")).sum()
    total_chars = len(group)

    if total_chars == 0:
        return 0

    rate = (backspace_count / total_chars)

    return round(rate, 2)

##Aalto

###Inter Key Timings Aalto

In [ ]:
def create_inter_key_timings_aalto(df):
    df = df.sort_values("PRESS_TIME")
    df["NEXT_PRESS"] = (df["PRESS_TIME"].shift(-1))
    df["INTER_KEY"] = (df["NEXT_PRESS"] - df["PRESS_TIME"])

    inter_keys = (df["INTER_KEY"].dropna().astype(int).tolist())
    inter_key_string = ",".join(map(str, inter_keys[:50]))

    return inter_keys, inter_key_string

###WPM Aalto

In [ ]:
def calculate_wpm_aalto(df):
    total_chars = len(df)
    duration_ms = (df["RELEASE_TIME"].max() - df["PRESS_TIME"].min())
    duration_min = (duration_ms / 1000 / 60)

    if duration_min == 0:
        return 0

    wpm = ((total_chars / 5) / duration_min)

    return round(wpm, 1)

###Backspace Rate Aalto

In [ ]:
def calculate_backspace_rate_aalto(df):
    backspace_count = ((df["LETTER"].astype(str).str.upper() == "BKSP")).sum()
    total_chars = len(df)

    if total_chars == 0:
        return 0

    rate = (backspace_count / total_chars)

    return round(rate, 2)

##Typing Variance

In [ ]:
def calculate_typing_variance(inter_keys):
    typing_variance = np.std(inter_keys)

    return round(typing_variance, 2)

##Stress Score

In [ ]:
def calculate_stress_score(backspace_rate, typing_variance, wpm):
    normalized_wpm = min(wpm / 120, 1)
    normalized_variance = (typing_variance / (typing_variance + 50))
    stress_score = (0.35 * backspace_rate + 0.4 * normalized_variance + 0.25 * (1 - normalized_wpm))
    stress_score = round(min(max(stress_score, 0), 1), 2)

    return stress_score

#Load Dataset

##Keyrecs Dataset

In [ ]:
df_keyrecs = pd.read_csv("https://raw.githubusercontent.com/raihanmeintaro/Dataset/refs/heads/main/Keysroke/KeyRecs%20Dataset.csv", low_memory=False)

Remove Unamed

In [ ]:
df_keyrecs = df_keyrecs.loc[:, ~df_keyrecs.columns.str.contains("^Unnamed")]

Rename

In [ ]:
df_keyrecs.columns = [
    "participant",
    "session",
    "key1",
    "key2",
    "DU_key1_key1",
    "DD",
    "DU",
    "UD",
    "UU"
]

Numeric

In [ ]:
timing_cols = [
    "DU_key1_key1",
    "DD",
    "DU",
    "UD",
    "UU"
]

In [ ]:
for col in timing_cols:
    df_keyrecs[col] = pd.to_numeric(df_keyrecs[col], errors='coerce')

df_keyrecs = df_keyrecs.dropna()

print(df_keyrecs.shape)

##Feature Engineering Keyrecs

In [ ]:
grouped = df_keyrecs.groupby(["participant", "session"])

keyrecs_sessions = []

for idx, ((participant, session), group) in enumerate(grouped):
    if len(group) < 20:
        continue

    inter_keys, inter_key_string = (create_inter_key_timings_keyrecs(group))

    wpm = calculate_wpm_keyrecs(group)

    if wpm == 0:
        continue

    backspace_rate = (calculate_backspace_rate_keyrecs(group))
    typing_variance = (calculate_typing_variance(inter_keys))
    stress_score = (calculate_stress_score(backspace_rate, typing_variance, wpm))

    keyrecs_sessions.append({

        "id":
        f"typ_{idx+1:05d}",

        "user_id":
        participant,

        "wpm":
        wpm,

        "typing_variance":
        typing_variance,

        "backspace_rate":
        backspace_rate,

        "inter_key_timings":
        inter_key_string,

        "stress_label":
        stress_score
    })

keyrecs_df = pd.DataFrame(keyrecs_sessions)

print(keyrecs_df.shape)
print(keyrecs_df["user_id"].nunique())

##Aalto Dataset

In [ ]:
!wget -O Keystrokes.zip \
"https://web.archive.org/web/20251127202912if_/https://userinterfaces.aalto.fi/136Mkeystrokes/data/Keystrokes.zip"

Extract Dataset

In [ ]:
with zipfile.ZipFile("Keystrokes.zip", 'r') as zip_ref:
    zip_ref.extractall("aalto_dataset")

print(os.listdir("aalto_dataset"))

Load Aalto Files

In [ ]:
DATASET_PATH = ("aalto_dataset/Keystrokes/files")
files_list = os.listdir(DATASET_PATH)

print("TOTAL FILES:", len(files_list))

##Feature Engineering Aalto

In [ ]:
aalto_sessions = []
MAX_FILES = 2500

for idx, file in enumerate(files_list[:MAX_FILES]):
    try:
        file_path = os.path.join(DATASET_PATH, file)

        df = pd.read_csv(file_path, sep="\t", engine="python", on_bad_lines="skip", encoding_errors="ignore")

        if len(df.columns) != 9:
            continue

        df.columns = [
            "PARTICIPANT_ID",
            "TEST_SECTION_ID",
            "SENTENCE",
            "USER_INPUT",
            "KEYSTROKE_ID",
            "PRESS_TIME",
            "RELEASE_TIME",
            "LETTER",
            "KEYCODE"
        ]

        df["PRESS_TIME"] = pd.to_numeric(df["PRESS_TIME"], errors='coerce')
        df["RELEASE_TIME"] = pd.to_numeric(df["RELEASE_TIME"], errors='coerce')

        df = df.dropna()

        if len(df) < 20:
            continue

        inter_keys, inter_key_string = (create_inter_key_timings_aalto(df))

        if len(inter_keys) < 10:
            continue

        wpm = calculate_wpm_aalto(df)

        if wpm == 0:
            continue

        backspace_rate = (calculate_backspace_rate_aalto(df))
        typing_variance = (calculate_typing_variance(inter_keys))
        stress_score = (calculate_stress_score(backspace_rate,typing_variance, wpm))

        aalto_sessions.append({
            "id":
            f"aalto_{idx+1:05d}",

            "user_id":
            str(
                df[
                    "PARTICIPANT_ID"
                ].iloc[0]
            ),

            "wpm":
            wpm,

            "typing_variance":
            typing_variance,

            "backspace_rate":
            backspace_rate,

            "inter_key_timings":
            inter_key_string,

            "stress_label":
            stress_score
        })

    except:
        continue

aalto_df = pd.DataFrame(aalto_sessions)

print(aalto_df.shape)
print(aalto_df["user_id"].nunique())

#Merge Real Dataset

In [ ]:
final_df = pd.concat([keyrecs_df, aalto_df], ignore_index=True)

print(final_df.shape)
print(final_df["user_id"].nunique())

final_df.head()

##Vizualization

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(final_df["stress_label"], bins=20)
plt.xlabel("Stress Label")
plt.ylabel("Count")
plt.title("Stress Label Distribution")
plt.show()

#Synthetic Dataset

In [ ]:
synthetic_df = pd.read_csv("https://raw.githubusercontent.com/raihanmeintaro/Dataset/main/Keysroke/Realistic_Minority_Augmentation_Users_v2.csv")

synthetic_df.shape

In [ ]:
synthetic_df["user_id"].value_counts()

##Select Minority Range from Synthetic

In [ ]:
minority_synth = synthetic_df[(synthetic_df["stress_label"] < 0.35) | (synthetic_df["stress_label"] > 0.60)].copy()

minority_synth.shape

In [ ]:
max_synthetic = int(len(final_df) * 0.30)

minority_synth = minority_synth.sample(n=min(len(minority_synth), max_synthetic), random_state=42)
print("Max Synthetic:", max_synthetic)
print("Used Sythetic:", len(minority_synth))

#Merge All Datasets

In [ ]:
final_balanced_df = pd.concat([final_df, minority_synth],ignore_index=True)

final_balanced_df.shape

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(final_balanced_df["stress_label"], bins=20)
plt.xlabel("Stress Label")
plt.ylabel("Count")
plt.title("Final Dataset Distribution")
plt.show()

In [ ]:
print("Real Data:",len(final_df))
print("Synthetic Data:", len(minority_synth))
print("Synthetic Percent:", round((len(minority_synth) / len(final_balanced_df)) * 100, 2), "%")

#Convert Dataset

In [ ]:
print(final_balanced_df.shape)
print(final_balanced_df["user_id"].nunique())

In [ ]:
final_balanced_df.shape

In [ ]:
final_balanced_df.to_csv("Keystroke_Stress_Dataset.csv", index=False)

# Data Dictionary

| Column | Type | Description |
|----------|----------|----------|
| id | string | Unique session identifier |
| user_id | string | Anonymous user identifier |
| wpm | float | Typing speed measured in Words Per Minute |
| typing_variance | float | Standard deviation of inter-key intervals (ms) |
| backspace_rate | float | Ratio of backspace key presses to total key presses |
| inter_key_timings | string | Sequence of inter-key intervals in milliseconds separated by commas |
| stress_label | float | Stress score ranging from 0.0 to 1.0 |

# Conclusion

This preprocessing phase successfully:

- Integrated the KeyRecs and Aalto datasets.
- Extracted session-level keystroke features.
- Standardized the dataset structure.
- Generated stress labels based on behavioral indicators.
- Applied controlled synthetic augmentation to improve target distribution coverage.

The resulting dataset is ready for Exploratory Data Analysis (EDA), feature selection, and machine learning model development for stress prediction using keystroke dynamics.